# Team 3: Spark Analytics
Full-dataset PySpark and Spark SQL analysis for Owl Analytics.


In [1]:
!pip -q install pyspark


In [2]:
from pyspark.sql import SparkSession, functions as F, types as T
spark=SparkSession.builder.appName('OwlAnalytics').getOrCreate()
print('Spark session started')


c:\Users\sikan\Documents\owl-analytics-completed\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark session started


In [3]:
path='data/clean/cleaned_market_data.csv'
df=(spark.read.option('header',True).option('inferSchema',True).csv(path)
    .withColumn('open_time',F.to_timestamp('open_time'))
    .withColumn('close_time',F.to_timestamp('close_time')))
print('Row count:',df.count()); print('Columns:',df.columns); df.printSchema(); df.show(5,truncate=False)


Row count: 9465
Columns: ['symbol', 'interval', 'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume', 'trade_count', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'price_range', 'price_change', 'percent_change', 'candle_direction']
root
 |-- symbol: string (nullable = true)
 |-- interval: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- quote_volume: double (nullable = true)
 |-- trade_count: integer (nullable = true)
 |-- taker_buy_base_volume: double (nullable = true)
 |-- taker_buy_quote_volume: double (nullable = true)
 |-- price_range: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- percent_change: double (nullable = true)
 |-- candle_direction: string (nullable = true)

+

In [4]:
df=(df.withColumn('price_range',F.col('high')-F.col('low'))
 .withColumn('price_change',F.col('close')-F.col('open'))
 .withColumn('percent_change',(F.col('close')-F.col('open'))/F.col('open')*100)
 .withColumn('candle_direction',F.when(F.col('close')>F.col('open'),'up').when(F.col('close')<F.col('open'),'down').otherwise('flat'))
 .withColumn('hour',F.hour('open_time')).withColumn('day_of_week',F.date_format('open_time','E'))
 .withColumn('date',F.to_date('open_time')))
df.createOrReplaceTempView('market_data'); print('Temporary SQL view created: market_data'); spark.sql('SELECT * FROM market_data LIMIT 10').show()


Temporary SQL view created: market_data
+-------+--------+-------------------+----------+----------+----------+----------+---------------+-------------------+--------------+-----------+---------------------+----------------------+--------------------+--------------------+--------------------+----------------+----+-----------+----------+
| symbol|interval|          open_time|      open|      high|       low|     close|         volume|         close_time|  quote_volume|trade_count|taker_buy_base_volume|taker_buy_quote_volume|         price_range|        price_change|      percent_change|candle_direction|hour|day_of_week|      date|
+-------+--------+-------------------+----------+----------+----------+----------+---------------+-------------------+--------------+-----------+---------------------+----------------------+--------------------+--------------------+--------------------+----------------+----+-----------+----------+
|ADAUSDT|      1h|2026-01-01 00:00:00|      0.45|0.45177176|0.4

## 1. Records and price overview by symbol


In [5]:
q="""SELECT symbol, COUNT(*) records, ROUND(AVG(close),6) avg_close, ROUND(MIN(low),6) min_low, ROUND(MAX(high),6) max_high FROM market_data GROUP BY symbol ORDER BY symbol"""
spark.sql(q).show(100,truncate=False)


+--------+-------+------------+------------+------------+
|symbol  |records|avg_close   |min_low     |max_high    |
+--------+-------+------------+------------+------------+
|ADAUSDT |954    |0.378576    |0.27937     |0.485954    |
|AVAXUSDT|947    |34.426689   |28.55787    |38.718726   |
|BNBUSDT |953    |515.450439  |470.771349  |567.63096   |
|BTCUSDT |947    |58824.886206|52960.153365|65863.348446|
|DOGEUSDT|953    |0.1175      |0.107045    |0.133857    |
|DOTUSDT |949    |6.182452    |5.390006    |7.28389     |
|ETHUSDT |933    |2736.986475 |2232.376758 |3153.224127 |
|LINKUSDT|951    |15.170498   |14.059916   |16.54899    |
|SOLUSDT |939    |181.515017  |146.265687  |204.089591  |
|XRPUSDT |939    |0.730131    |0.596425    |0.905257    |
+--------+-------+------------+------------+------------+



## 2. Volatility ranking


In [6]:
q="""SELECT symbol, ROUND(AVG(price_range),6) avg_price_range, ROUND(STDDEV(percent_change),6) percent_change_stddev FROM market_data GROUP BY symbol ORDER BY percent_change_stddev DESC"""
spark.sql(q).show(100,truncate=False)


+--------+---------------+---------------------+
|symbol  |avg_price_range|percent_change_stddev|
+--------+---------------+---------------------+
|SOLUSDT |1.442815       |0.604907             |
|ETHUSDT |22.166017      |0.604255             |
|DOTUSDT |0.049369       |0.60032              |
|XRPUSDT |0.005875       |0.600168             |
|DOGEUSDT|9.44E-4        |0.599845             |
|BTCUSDT |469.140775     |0.598743             |
|AVAXUSDT|0.272118       |0.594574             |
|LINKUSDT|0.119996       |0.591686             |
|ADAUSDT |0.002932       |0.580698             |
|BNBUSDT |4.125708       |0.580535             |
+--------+---------------+---------------------+



## 3. Trading activity ranking


In [7]:
q="""SELECT symbol, ROUND(SUM(volume),2) total_volume, SUM(trade_count) total_trades, ROUND(AVG(trade_count),2) avg_trades FROM market_data GROUP BY symbol ORDER BY total_trades DESC"""
spark.sql(q).show(100,truncate=False)


+--------+--------------+------------+----------+
|symbol  |total_volume  |total_trades|avg_trades|
+--------+--------------+------------+----------+
|AVAXUSDT|1.621558678E7 |4788415     |5056.4    |
|DOTUSDT |3.619310633E7 |4780656     |5037.57   |
|BTCUSDT |384423.85     |4752127     |5018.09   |
|BNBUSDT |4302551.19    |4748982     |4983.19   |
|XRPUSDT |1.2101947029E8|4718350     |5024.87   |
|DOGEUSDT|2.7544950775E8|4717699     |4950.37   |
|ADAUSDT |1.4199426806E8|4716938     |4944.38   |
|SOLUSDT |7784025.69    |4713670     |5019.88   |
|ETHUSDT |1669707.5     |4681494     |5017.68   |
|LINKUSDT|2.481123089E7 |4676253     |4917.2    |
+--------+--------------+------------+----------+



## 4. Candle direction counts


In [8]:
q="""SELECT symbol, candle_direction, COUNT(*) candles FROM market_data GROUP BY symbol,candle_direction ORDER BY symbol,candles DESC"""
spark.sql(q).show(100,truncate=False)


+--------+----------------+-------+
|symbol  |candle_direction|candles|
+--------+----------------+-------+
|ADAUSDT |down            |499    |
|ADAUSDT |up              |455    |
|AVAXUSDT|down            |486    |
|AVAXUSDT|up              |461    |
|BNBUSDT |up              |477    |
|BNBUSDT |down            |476    |
|BTCUSDT |up              |482    |
|BTCUSDT |down            |465    |
|DOGEUSDT|up              |481    |
|DOGEUSDT|down            |472    |
|DOTUSDT |down            |497    |
|DOTUSDT |up              |452    |
|ETHUSDT |up              |470    |
|ETHUSDT |down            |463    |
|LINKUSDT|up              |498    |
|LINKUSDT|down            |453    |
|SOLUSDT |up              |485    |
|SOLUSDT |down            |454    |
|XRPUSDT |up              |509    |
|XRPUSDT |down            |430    |
+--------+----------------+-------+



## 5. Strongest positive and negative hourly moves


In [9]:
q="""SELECT symbol, open_time, ROUND(percent_change,4) percent_change, candle_direction FROM market_data ORDER BY ABS(percent_change) DESC LIMIT 20"""
spark.sql(q).show(100,truncate=False)


+--------+-------------------+--------------+----------------+
|symbol  |open_time          |percent_change|candle_direction|
+--------+-------------------+--------------+----------------+
|ADAUSDT |2026-01-28 19:00:00|-2.4907       |down            |
|XRPUSDT |2026-02-08 15:00:00|-2.2315       |down            |
|SOLUSDT |2026-02-06 10:00:00|-2.2308       |down            |
|ADAUSDT |2026-02-03 06:00:00|-2.2032       |down            |
|ETHUSDT |2026-01-21 19:00:00|-2.1356       |down            |
|DOGEUSDT|2026-01-19 12:00:00|-2.0059       |down            |
|SOLUSDT |2026-01-08 12:00:00|1.9681        |up              |
|LINKUSDT|2026-02-05 09:00:00|-1.951        |down            |
|SOLUSDT |2026-01-22 18:00:00|1.9347        |up              |
|BNBUSDT |2026-02-04 08:00:00|-1.9085       |down            |
|DOTUSDT |2026-01-27 21:00:00|-1.8849       |down            |
|XRPUSDT |2026-02-09 17:00:00|-1.8822       |down            |
|DOGEUSDT|2026-01-31 18:00:00|1.8745        |up        

## 6. Activity by hour of day


In [10]:
q="""SELECT hour, SUM(trade_count) total_trades, ROUND(SUM(volume),2) total_volume FROM market_data GROUP BY hour ORDER BY total_trades DESC"""
spark.sql(q).show(100,truncate=False)


+----+------------+-------------+
|hour|total_trades|total_volume |
+----+------------+-------------+
|1   |2046284     |2.603237727E7|
|7   |2020785     |2.533070164E7|
|22  |2008247     |2.64919329E7 |
|3   |2008039     |2.776344764E7|
|4   |2004354     |2.66872584E7 |
|14  |2004106     |2.532528689E7|
|12  |2002274     |2.606632074E7|
|13  |1994183     |2.642103047E7|
|6   |1990391     |2.727902868E7|
|2   |1987452     |2.720791744E7|
|17  |1986709     |2.678273824E7|
|11  |1983199     |2.617245414E7|
|10  |1976133     |2.722200475E7|
|20  |1967927     |2.493286554E7|
|0   |1962804     |2.651855809E7|
|15  |1962234     |2.607182827E7|
|16  |1947764     |2.696663502E7|
|9   |1942464     |2.521329455E7|
|21  |1935986     |2.609470783E7|
|5   |1931182     |2.516281454E7|
|8   |1920793     |2.628899686E7|
|18  |1916653     |2.588967526E7|
|19  |1905918     |2.667193958E7|
|23  |1888703     |2.523006361E7|
+----+------------+-------------+



## 7. Activity by day of week


In [11]:
q="""SELECT day_of_week, SUM(trade_count) total_trades, ROUND(AVG(ABS(percent_change)),4) avg_absolute_change FROM market_data GROUP BY day_of_week ORDER BY total_trades DESC"""
spark.sql(q).show(100,truncate=False)


+-----------+------------+-------------------+
|day_of_week|total_trades|avg_absolute_change|
+-----------+------------+-------------------+
|Sat        |6915927     |0.4717             |
|Fri        |6843544     |0.4637             |
|Tue        |6809808     |0.4829             |
|Thu        |6792031     |0.4905             |
|Mon        |6771898     |0.4717             |
|Sun        |6768506     |0.4729             |
|Wed        |6392870     |0.4732             |
+-----------+------------+-------------------+



## 8. Daily market movement


In [12]:
q="""SELECT date, ROUND(AVG(percent_change),4) avg_percent_change, ROUND(SUM(volume),2) total_volume FROM market_data GROUP BY date ORDER BY date"""
spark.sql(q).show(100,truncate=False)


+----------+------------------+-------------+
|date      |avg_percent_change|total_volume |
+----------+------------------+-------------+
|2026-01-01|0.0461            |1.520720926E7|
|2026-01-02|0.0261            |1.638415821E7|
|2026-01-03|0.0161            |1.546681704E7|
|2026-01-04|-0.0164           |1.490565843E7|
|2026-01-05|0.0741            |1.603741238E7|
|2026-01-06|-0.0356           |1.403735588E7|
|2026-01-07|-0.0344           |1.554612368E7|
|2026-01-08|-0.0076           |1.506250247E7|
|2026-01-09|0.0555            |1.525172594E7|
|2026-01-10|-0.0369           |1.433891404E7|
|2026-01-11|0.0342            |1.468777564E7|
|2026-01-12|0.0071            |1.518750807E7|
|2026-01-13|-0.0044           |1.536896854E7|
|2026-01-14|0.0318            |1.517781751E7|
|2026-01-15|-0.0564           |1.583501441E7|
|2026-01-16|0.0221            |1.373852481E7|
|2026-01-17|-0.0803           |1.47677775E7 |
|2026-01-18|-0.0164           |1.552608367E7|
|2026-01-19|0.0298            |1.5

In [14]:
summary=spark.sql("""SELECT symbol, COUNT(*) records, ROUND(AVG(close),6) average_close, ROUND(AVG(price_range),6) average_price_range, ROUND(STDDEV(percent_change),6) volatility_score, SUM(trade_count) total_trades, ROUND(SUM(volume),2) total_volume, ROUND(AVG(percent_change),6) average_percent_change FROM market_data GROUP BY symbol ORDER BY volatility_score DESC""")
summary.show(truncate=False)
# Spark folder export disabled because of a local Windows/Java issue.
# The results are still saved below as a standard CSV file.

summary.toPandas().to_csv(
    "results/spark_market_summary.csv",
    index=False
)

print("Saved results/spark_market_summary.csv")


+--------+-------+-------------+-------------------+----------------+------------+--------------+----------------------+
|symbol  |records|average_close|average_price_range|volatility_score|total_trades|total_volume  |average_percent_change|
+--------+-------+-------------+-------------------+----------------+------------+--------------+----------------------+
|SOLUSDT |939    |181.515017   |1.442815           |0.604907        |4713670     |7784025.69    |0.030091              |
|ETHUSDT |933    |2736.986475  |22.166017          |0.604255        |4681494     |1669707.5     |-0.006758             |
|DOTUSDT |949    |6.182452     |0.049369           |0.60032         |4780656     |3.619310633E7 |-0.016534             |
|XRPUSDT |939    |0.730131     |0.005875           |0.600168        |4718350     |1.2101947029E8|0.034364              |
|DOGEUSDT|953    |0.1175       |9.44E-4            |0.599845        |4717699     |2.7544950775E8|0.004878              |
|BTCUSDT |947    |58824.886206 |

c:\Users\sikan\Documents\owl-analytics-completed\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\sikan\Documents\owl-analytics-completed\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Saved results/spark_market_summary.csv


## Interpretation
The final ranking separates volatility from activity. A market can have a high trading count without having the largest percentage swings. The hourly and weekday queries show when the dataset was most active, while the candle-direction and extreme-move queries identify directional behaviour and outliers. Exact rankings depend on the API extraction time, so the saved CSV is the authoritative run output.
